# 13 — Capstone: CA Assistant

> **ICAN CA Training — Generative AI & RAG (6 hours).** This notebook is part of a 14-notebook curriculum. All data is synthetic. Confidential client data must not be used with public APIs without engagement-letter authority. AI output must always be verified by a qualified professional.


## Learning objectives
1. Bring it all together: documents + tables + graph + agent.
2. Produce a short professional-style report with citations and a limitations footer.
3. Adapt the assistant to one of five capstone scenarios.


## Choose a scenario

Pick one and run it through the assistant below:
1. **Audit risk assistant** — "What are the top audit risks and which evidence supports each?"
2. **Tax compliance Q&A** — "What is our tax-compliance status and what action is required by 2082-06-30?"
3. **Financial statement review** — "Walk me through the loan disclosures vs the loan agreement."
4. **Internal control review** — "List approval-threshold violations in the purchases data and the policy rule each violates."
5. **Related-party transaction assistant** — "Map every related-party transaction to its Board approval (if any)."


In [ ]:
# --- Bootstrap (don't edit) ---
# Adds the project root to sys.path so we can do `from src.xxx import yyy`.
import sys, os
from pathlib import Path
ROOT = Path.cwd()
# Walk up until we find the project root (folder that contains src/)
for _ in range(4):
    if (ROOT / 'src').exists() and (ROOT / 'requirements.txt').exists():
        break
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
os.chdir(ROOT)
print('Project root:', ROOT)


In [ ]:
import pandas as pd
from datetime import datetime
from pathlib import Path
from src.rag_utils import build_store_from_folder, rag_answer
from src.graph_utils import build_finance_graph
from src.agentic_rag_utils import safe_calc, make_table_tool, make_graph_tool, run_agent
from src.llm_client import ask_llm
from src.evaluation_utils import SAFE_USE_CHECKLIST

store = build_store_from_folder('data/generated/pdf')
G = build_finance_graph()
tables = {
    'sales':     pd.read_csv('data/generated/csv/01_sales_transactions.csv'),
    'purchases': pd.read_csv('data/generated/csv/02_purchase_transactions.csv'),
    'journals':  pd.read_csv('data/generated/csv/03_journal_entries.csv'),
    'vendors':   pd.read_csv('data/generated/csv/04_vendor_master.csv'),
    'rpt':       pd.read_excel('data/generated/xlsx/10_related_party_transactions.xlsx'),
    'budget':    pd.read_csv('data/generated/csv/08_budget_vs_actual.csv'),
}

def search_documents(q):
    ans, hits = rag_answer(q, store, k=4, return_hits=True)
    sources = ', '.join(f"{h['metadata'].get('source')}#p{h['metadata'].get('page','-')}" for h in hits)
    return f'{ans}\n[sources: {sources}]'

tools = {
    'search_documents': search_documents,
    'query_table':      make_table_tool(tables),
    'query_graph':      make_graph_tool(G),
    'calculate':        lambda x: str(safe_calc(x)),
}
print('Capstone assistant ready. Tools:', list(tools))

## 13.1 — Run the capstone task

In [ ]:
# === EDIT THIS ===
scenario_title = 'Internal Control Review — Approval-Threshold Compliance'
scenario_task  = (
    'For the synthetic company Himal Trading, list every purchase invoice above NPR 5,00,000 '
    'from the purchases table; check whether the procurement policy required dual approval '
    'for each; check whether the vendor is a related party using the graph; and produce a '
    'short findings list with citations to the policy and ledger evidence.'
)

trace = run_agent(scenario_task, tools=tools, max_steps=6, verbose=True)
print('\n=== FINAL ANSWER ===')
print(trace.answer)

## 13.2 — Render a professional-style report

In [ ]:
report_prompt = f'''Convert the following draft findings into a short report for a Chartered 
Accountant. Sections: (1) Background, (2) Procedures performed, (3) Findings, (4) Recommended 
actions, (5) Limitations of this AI-assisted review (mention RAG hallucination risk, synthetic data, 
need for human verification). Keep it under 350 words.\n\n'''
report_prompt += f'Title: {scenario_title}\n\nDraft findings:\n{trace.answer}'
report = ask_llm(report_prompt, system='You are a senior CA writing for a partner.', temperature=0.1)
print(report)

## 13.3 — Save the report (and a quick log of the agent trace)

In [ ]:
out_dir = Path('outputs/reports')
out_dir.mkdir(parents=True, exist_ok=True)
stamp = datetime.now().strftime('%Y%m%d_%H%M%S')
report_path = out_dir / f'{stamp}_capstone_report.md'
trace_path  = out_dir / f'{stamp}_capstone_trace.md'
report_path.write_text(
    f'# {scenario_title}\n\n*Generated {stamp}*\n\n{report}\n\n'
    f'---\n\n## Limitations & safe-use checklist\n\n{SAFE_USE_CHECKLIST}\n',
    encoding='utf-8',
)
trace_lines = []
for s in trace.steps:
    trace_lines.append(f"### Step {s['step']} — action: {s.get('action')}")
    trace_lines.append('**Input:** ' + (s.get('input') or '_(none)_'))
    trace_lines.append('**Observation:**')
    trace_lines.append('```\n' + (s.get('observation') or '') + '\n```')
trace_path.write_text(
    f'# Capstone agent trace\n\n{scenario_task}\n\n' + '\n\n'.join(trace_lines),
    encoding='utf-8',
)
print('Wrote:')
print('  ', report_path)
print('  ', trace_path)

## Exercise

1. Edit the scenario above to one of the four other capstones and re-run.
2. Add an extra section to the report: *"Three questions I would ask management."*
3. Compare the AI's findings list with what you would write manually — what did the AI miss? What did it get wrong?


## ⚠️ Closing professional caution

What you have built today is a **research assistant**, not an audit conclusion. Every figure, citation, and policy link must be re-verified by a qualified professional before it informs any client-facing communication. The synthetic data is benign; real client data demands the Safe-Use Checklist below.

In [ ]:
print(SAFE_USE_CHECKLIST)